# 03 — PPO attack environment

This notebook uses a synthetic image and deterministic mock victim, so it runs without DeepfakeBench. The real detector uses the same `predict_image` contract.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import numpy as np
from PIL import Image
from stable_baselines3.common.env_checker import check_env

from ppo.environment import DeepfakeAttackEnv
from src.ppo.mock_detector import MockDetector

demo_dir = PROJECT_ROOT / "outputs/notebook_demo"
demo_dir.mkdir(parents=True, exist_ok=True)
image_path = demo_dir / "synthetic_fake.png"
x = np.linspace(96, 240, 224, dtype=np.uint8)
pixels = np.tile(x, (160, 1))
Image.fromarray(np.stack([pixels] * 3, axis=2)).save(image_path)

env = DeepfakeAttackEnv([image_path], MockDetector(), max_steps=5, seed=42)
check_env(env, warn=True)
print("Environment check passed", env.action_space, env.observation_space)

In [ ]:
observation, reset_info = env.reset(seed=42)
print("reset", observation, reset_info)
for action in ([5, 3], [3, 2], [1, 1]):
    observation, reward, terminated, truncated, info = env.step(np.array(action))
    print(f"action={action} reward={reward:.4f} obs={observation}")
    print(info)
    if terminated or truncated:
        break

In [ ]:
assert observation.shape == (6,) and observation.dtype == np.float32
assert env.original_image is not env.current_image
assert 0 <= info["ssim"] <= 1 and 0 <= info["perturbation"] <= 1
print("Reward direction: positive confidence gain means P(FAKE) decreased.")